# v4 语义复发保护与残留修复 v2

日期：2026-11-03。依据 `formal_30_v4_semantic_closure_v1_results_20261103_145253`（290 文件）。上一模型批两项均未通过；TaskManager 第三次重引第二次已消除的隐藏 tasks 依赖，旧保护门没有拒收。必须把真实复发加进回归，不能只增加 Prompt 或反复重抽。

新批次 `formal_30_v4_semantic_closure_v2`，规则集 `residual_v2`，仍为 TaskManager／MonitoringDashboard 重复开发样本，不是独立测试集。旧结果、原始 San、repair history 和评分保持。

## 实现与边界

- `repair_scheduler.py`：按 code／location／message／次数比较已列入保护的语义发现（隐藏依赖、computed 实例调用、生命周期、dataTypes、prop defaults）。总错误数减少不能抵消新增问题。拒收保留原候选及失败状态，坏文件和用量留证。它不是所有行为的等价证明，未知错误／警告未自动扩展。
- `residual_semantics.cjs/.py`：只在 semantic_rule_set=residual_v2 时处理自包含 computed 方法和源子组件 props schema，接入确定性计划及 LLM 返回后。computed 调用改用脚本 helper，原实例方法与实参保留；不接受外部闭包、this、未知副作用或参数变更。允许原有局部 Date 计算，但不将时钟读取称为数学纯函数。
- dataTypes 依据完整源 inline Vue props 和实际组件注册映射，补真实字段／已知类型，不填空 schema、不虚构无源父 props。已有 schema、未知类型／不明确注册不猜改，default 等其它契约独立验证。
- `repair_prompt.py`：missing_data_types 混合残留不再阻断聚焦路由，单一隐藏依赖／生命周期也可聚焦；带真实源 props 以及源 hook→方法递归调用链。未自动搬动 localStorage／watcher／Notification 初始化；复杂时序交给有证据的模型修复后重新评估。

generation／服务器 Prompt 和验证器未改。run_batch_pipeline 验证、传递并记录规则集；未知规则或非 bounded 策略配 residual_v2 会拒绝。原 bounded_v1 不默认应用新的 AST 转换。

## 无模型回放和正反例

运行 `python -m migration_pipeline.replay_formal_30_v4_residual_v2`，原快照 290 文件回放前后验证，输出仅写独立分析目录。真实冻结第 2→3 次 TaskManager 代码重放被语义门拒收，final 恢复第 2 次且仍失败；第三次坏候选保留。

| 冻结最终输入 | 新规则后剩余 | 初始 DOM | 结论 |
| --- | --- | ---: | --- |
| TaskManager | hidden_template_reactive_dependency | 0.9889 | 仍失败 |
| MonitoringDashboard | lifecycle_behavior_not_landed | 0.9605 | 仍失败 |

TaskManager computed／dataTypes 问题在此输入上消除；MonitoringDashboard getX/getY 上下文错误消除。不能据此宣称新模型迁移通过。后续仍需 TaskManager 的显式 tasks 依赖、MonitoringDashboard 源初始化时序修复和交互验证。

准备的聚焦 Prompt 为 56,256／66,701 字符，来源及提示哈希见 `data/experiments/analysis/formal_30_v4_residual_v2_20261103/replay.json`。与上一模型批约 44 万字符仅作文本长度观察，没有新 token 消耗或成功率证据。

新增语义保护 7 项、residual-v2 12 项测试。包括总数减少仍有新错误、同族不同发现、实际冻结回退；拒绝外部闭包／参数变更／未知数学调用、计算边界、固定时钟 Date 等价、schema 类型及注册拒修、Prompt 源契约、真实 repair 返回后处理和 opt-in。

## 新版本运行（尚未调用模型）

新协议 `data/experiments/protocols/formal_30_v4_semantic_closure_v2_frozen_20261103/PROTOCOL.md`，完整测试、离线回放、dry-run、代码与配置哈希在首次调用前封存。旧包不修改，旧 live 基线与新实现不同属预期。

仍用 bounded_prepass_v1 调度：每样本最多 3 次 LLM repair，每次前最多 4 步归一化。qwen/qwen-plus，温度 0.2，600／660／660 秒超时。多项规则／Prompt／候选门同时改变且重新生成，不作单因素因果或等实际 token 结论。

停止旧服务。窗口 A 设好 LLM_API_KEY 后运行 `python -m migration_pipeline.run_formal_30_v4_semantic_closure_v2 --server`；窗口 B 运行一次 `python -m migration_pipeline.run_formal_30_v4_semantic_closure_v2 --execute`。独立新默认目录，不加 --force，运行结束先冻结再诊断。

当前无本批模型 summary，未修改测试页、未手改旧生成 San、未 commit／push。完整 30 项回归和其它三个失败项仍未结案，v5 探索性评估等待。

实际预运行验证：269/269 测试通过（66.053 秒），含本次新增 19 项；新协议封存 270 份 payload、278 项 live 哈希校验通过；dry-run 和离线回放通过，模型请求数 0。旧快照 290 项复核通过。

## 实际模型运行结果（2026-11-03）

运行 15:32:26–16:03:55（UTC+8）。2 完成、1 自动通过、1 失败、0 执行错误。先冻结后详细分析：`data/experiments/snapshots/formal_30_v4_semantic_closure_v2_results_20261103_160541`，297 文件验证，冻结时协议 278 项 live 哈希通过。实际服务器超时配置未由文件校验或健康检查独立证明。

| 样本 | LLM repair 调用 | 确定性步骤（均接受） | 最终 DOM 相似度 | 自动结论 |
| --- | ---: | ---: | ---: | --- |
| TaskManager | 2 | 2 | 0.9889 | 静态和初始 DOM 通过，人工待验 |
| MonitoringDashboard | 3 | 1 | 0.9825 | event_arguments_not_landed／lifecycle_behavior_not_landed 未通过 |

生成 203,966、repair 449,644、合计 653,610 tokens。TaskManager 764.781 秒、MonitoringDashboard 1124.688 秒，均为样本总用时。前 v1 使用 6 次 LLM repair、本次 5 次，上限相同；生成随机性及规则／Prompt／候选门同时不同，不作单因素因果或等实际消耗推断。

TaskManager：两步预处理记录真实 inline props schema、self／drag／methods／registry 修复和自包含 computed helper；第一次 LLM 的后处理也提取 helper。第一轮后剩事件语法和隐藏依赖，第二轮已无静态错误并停止。提示长度 444481／57136 字符，第二轮为聚焦。没有拒收或回退，不择优挪用旧通过 San。仍有 migration_hints_not_landed 警告，未做 automatic interaction 或本轮人工交互验证。

注意：两个人工测试页当前仍引用 2026-10-29 的旧 TaskManager 快照，未切换至本批。不能直接在当前页面确认后归属为新产物通过。后续应只换引用到本批源 Vue 和最终 San，再人工复查设置、新建／编辑／关闭、模态内部点击、拖拽排序、筛选计数及任务状态联动。

MonitoringDashboard：新 schema 和计算规则实际执行；三次后分别剩 4／3／2 类错误。最终 onKPIClick(kpi) 被改成无业务参数绑定，父方法读 e.index，子组件 fire('click') 却不携带 index，事件链不一致；当前窄规则未覆盖带参数但 payload 错配的这种改写。

生命周期不是动作全部丢失：初始化链、startTimeUpdate、startDataSimulation 被搬到了 inited，源 Vue 要求 mounted→attached。需恢复源时机和顺序，不能直接豁免 checker。最终 DOM 0.9825，前两次 0.9912，均在现有阈值内，不以相似度数值微降自动拒收。

本轮无候选退化拒收；保护机制已由前期冻结坏候选回放和正反例验证，本次不额外声称演示回退。第二次 MonitoringDashboard repair 仍完整 Prompt（477646 字符），因为当时缺少 missing_lifecycle_mapping，不在聚焦集合；第三次才聚焦（63853 字符）。这是后续路由审查点，不是单独失败因果结论。

详细证据：`data/experiments/analysis/formal_30_v4_semantic_closure_v2_20261103_160541/REPORT.md`、summary.json；复现 `python -m migration_pipeline.analyze_formal_30_v4_semantic_closure_v2 --snapshot data/experiments/snapshots/formal_30_v4_semantic_closure_v2_results_20261103_160541`。最终代码、路径、静态／视觉评估、规则集、用量与历史已核对一致。

下一步（未实施）：保留 TaskManager 通过产物并切换测试页人工验证，不再重生它；MonitoringDashboard 用本次失败快照补 KPI payload／生命周期阶段正反例，修通用规则和聚焦边界后建新版本仅验证失败项。本次只冻结、诊断和记录，未修改运行期规则／Prompt／验证器、测试页或生成 San，未 commit／push。完整 30 项和其它三个失败事项仍未结案，v5 继续等待。

## TaskManager 人工发现 textarea 缺陷及独立修复（2026-11-03）

用户反馈新建任务的任务描述中显示代码；核对本轮冻结 San 为 `<textarea>{= formData.description =}</textarea>`，源 Vue 为 textarea v-model。San 应在 value 属性中使用双向绑定。原自动通过记录不变，但人工状态为发现缺陷、未通过；说明初始 DOM 及旧静态检查漏掉隐藏模态框。

新增 scoped textarea 分析／修复、invalid_san_textarea_binding 检测、repair 预处理及返回后规则、Prompt 约束和语义退化保护。只修复正文完整为可赋值路径标记且没有既有 value 的明确模式；字面文字／歧义内容不覆盖，不顺带改其它 HTML。

用户进一步确认英文提示为“设置已更新：defaultView”，更改设置时出现。Vue 与 San 均使用 `设置已更新：${key}`，本次判为源已有文案而非新增迁移缺陷，不改源组件或随意翻译。

独立候选 `data/experiments/manual_followups/formal_30_v4_semantic_closure_v2_TaskManager_textarea_v1_20261103/TaskManager_textarea_fixed.san`，13 文件验证；原 297 文件快照不变。新检查下原件报 textarea 错误，候选静态通过；无模型调用，不修改原 summary、不回填原自动预算。candidate_manual_status=pending_retest。

人工反馈机器记录及说明在本轮分析目录的 `TaskManager_manual_feedback_20261103.json`、`TaskManager_manual_textarea_followup_REPORT.md`。San 页面只切 componentPath 到候选，Vue 仍用本轮冻结源，其他页面代码保留。请用户重新检查描述初始为空、输入／保存／编辑回显以及原交互。没有助手浏览器人工通过结论。

新规则／checker 变更使旧 live 代码基线不再匹配；旧冻结包不改，后续模型运行需要新协议。

实际验证：279/279 测试通过（85.535 秒），其中 textarea 边界 10 项；候选包 13 文件及原快照 297 文件校验通过。用户人工复验待确认，未启动模型。

## TaskManager 修复候选人工复验通过（2026-11-03）

用户在当前 San 测试页复验后回复‘通过’。确认对象是独立候选 `data/experiments/manual_followups/formal_30_v4_semantic_closure_v2_TaskManager_textarea_v1_20261103/TaskManager_textarea_fixed.san`，SHA-256：`8286510708e7a273388b205af6217a60f527a535786cdb25ec67f81fdc771b46`，状态为 passed_by_user。依据仅为用户整体确认，助手未执行浏览器交互测试，不补造逐项观察。

人工验收记录位于本轮分析目录的 `TaskManager_manual_acceptance_20261103.json`；`TaskManager_manual_feedback_20261103.json` 已更新候选状态。原件 textarea 缺陷及此前待复验历史保留；已冻结的候选包 manifest 和 SHA256SUMS 不修改，由独立验收记录补充后续状态。

这是人工反馈后通过通用确定性规则得到的修复候选，不回填原三轮预算内的纯自动结果。原始批次仍为 2 项完成、1 项自动通过、1 项失败；原始快照与 batch summary 不变。本次额外模型调用为 0。

下一待办：MonitoringDashboard 的 KPI 事件参数链与生命周期阶段／顺序仍未解决，需以冻结失败产物补通用规则与正反例，再冻结新版本验证；完整 30 项回归仍未完成。

## MonitoringDashboard 通用事件／阶段闭环及离线验证（2026-11-03）

在原 v2 冻结失败产物上确认：源 onKPIClick(kpi) 被改成读取 e.index，但子组件 fire('click') 没有相应索引；源 mounted 初始化与计时器被提前到 inited；本次还确认设置 watcher 缺少源 localStorage 持久化写入。旧批次的两项错误和分数不改，新检查器另外明确报告 source_mounted_order_not_landed、source_watch_persistence_not_landed。

新增 source_phase_semantics.cjs，以源 Vue AST、唯一事件和循环上下文、完整方法主体及生命周期语句序列为证据，成对修复参数与方法、移动已证明的挂载调用后缀、补回其余主体完全一致的 watcher 所缺末尾存储写入。无组件名硬编码；不明绑定、额外副作用、复杂初始化、未知源钩子或已有冲突写入不猜改。修复入口在 repair 确定性预处理／返回后，generation 及服务器 Prompt 未改。

repair 聚焦 Prompt、验证器和候选语义退化保护同步更新；缺失生命周期映射和新增诊断纳入聚焦，按源阶段／顺序及 watcher 副作用处理，不能只改 Prompt 或放宽校验。12 项新增测试覆盖真实原件、实际 San 监听器、调用顺序、拒修边界及回归门；完整 291/291 测试通过，116.043 秒。

独立离线证据包：data/experiments/analysis/formal_30_v4_source_phase_closure_v1_20261103，12 文件校验通过。三项规则后静态错误清零、San 编译及初始 DOM 通过、候选门通过，子组件和 CSS 不改。首次诊断相似度 0.9912，封存重放前／后为 0.9912／0.9825；不挑高值，随机输入未固定，不作稳定提升推断。候选 SHA-256：28290086017b0568aa39db3936390668ce8ce1b08048a1f27050a424db77c968。

详见本轮原分析目录的 MonitoringDashboard_source_phase_closure_REPORT.md 与新离线包 replay.json。重放命令：python -m migration_pipeline.replay_formal_30_v4_source_phase_closure。模型调用 0，无浏览器交互测试，不换测试页引用、不改原 batch summary、不 commit／push；TaskManager 已通过候选 13 文件和原始快照 297 文件复核通过。

本次是已知失败代码的离线修复闭环，不是新模型迁移通过或独立测试。原批次仍 1/2 自动通过。下一步另建冻结协议，仅新生成／验证 MonitoringDashboard（最多 3 次 LLM repair），由操作者启动模型运行；不重新生成当前 TaskManager 人工通过产物。新运行先冻结再检查，仍需人工交互与后续完整 30 项回归。旧 live 基线已不同，不改旧包来绕过校验。

## semantic closure v3 单样本预运行协议冻结（2026-11-03）

新批次 formal_30_v4_semantic_closure_v3，仅包含 MonitoringDashboard；协议为 data/experiments/protocols/formal_30_v4_semantic_closure_v3_frozen_20261103/PROTOCOL.md。使用与原冻结包一致的源 Vue，重新提取 SSM 并生成，再在本轮候选上迭代；不读取离线修复 San 作为初始输出或答案。TaskManager 排除，保留其已有人工通过候选及验收。

预算仍为最多 3 次 LLM repair，每次之前最多 4 步确定性修复；bounded_prepass_v1、residual_v2；qwen/qwen-plus、temperature 0.2、上游 600 秒、generation／repair 客户端各 660 秒。独立输出及 summary checkpoint，不传 --force。相对旧冻结基线变化为 repair.py、validate.py、bounded_semantics.cjs、repair_prompt.py、repair_scheduler.py，另外封存新增规则／脚本／测试。新检查器与规则同时变化，不作单因素归因。

本次预运行完整 297/297 测试通过，115.888 秒，其中新增 6 项单样本配置／启动边界测试；本次再次离线重放静态与 DOM 通过，相似度 0.9912，保存为新预运行日志，不替换先前离线封存的 0.9825。原始结果快照 297 文件、原离线闭环包 12 文件、TaskManager 候选包 13 文件复核通过。新协议封存 282 项 payload，共 304 项校验，含 live 校验通过。

启动脚本在启动服务／批处理前验证 live 哈希，并在新进程导入配置前设置固定环境。窗口 A：停止旧服务、设置 LLM_API_KEY 后执行 python -m migration_pipeline.run_formal_30_v4_semantic_closure_v3 --server；窗口 B：python -m migration_pipeline.run_formal_30_v4_semantic_closure_v3 --execute。冻结脚本与 dry-run 没有模型调用，尚无本轮 summary 或迁移通过结论；真实服务器加载状态不能由磁盘校验独立证明。

运行后先执行 python -m migration_pipeline.freeze_formal_30_v4_semantic_closure_v3_results 封存，再分析和安排人工交互验证。当前两页仍保留 TaskManager 引用，本次不切页，不把当前页面的通过归到新 MonitoringDashboard。未 commit／push，旧包／旧分数不修改；完整 30 项开发回归和其它失败项仍待完成，探索性评估继续等待。

## semantic closure v3 实际结果与回退诊断（2026-11-03）

用户完成本轮运行后，先冻结为 data/experiments/snapshots/formal_30_v4_semantic_closure_v3_results_20261103_175401，315 文件验证，协议 304 项 live 校验通过。运行 17:44:11–17:53:44（UTC+8），572.797 秒；MonitoringDashboard 1 完成、0 通过、1 失败、0 执行异常。一次确定性步骤接受，一次 LLM 修复因候选门拒收而回退，并非耗尽 3 次预算。

生成 106,522、修复 133,611、合计 240,133 tokens，拒收用量全部计入。最终保留 MonitoringDashboard_normalized_1.san，仍有非法内联标识符／编译错误、DOM 相似度 0；拒收 MonitoringDashboard_normalized_1_repaired.san 单独保存，可编译、初始 DOM 相似度 0.9912，但有 7 条静态错误（事件语法、computed 调实例方法、4 条隐藏依赖、KPI 参数绑定），不得改记通过。

保护门因 4 条 hidden_template_reactive_dependency 新出现而拒收。只读范围对照及隔离复检确认，RealTimeChart.getX/getY、RequestChart.getX/getHeight 方法主体以及对应渲染调用在前后相同。原件 const real-time-chart／request-chart 非法，inline 检查器只枚举合法 const 标识符，导致旧范围漏检；修好变量名之后，同样缺陷才可见。诊断差集不是实际新缺陷的充分证据；当前门未表达检测覆盖／不可分析状态。这解释四项回退依据，不代表候选其它错误无关紧要，也不改变失败计分。

本轮新增源参数／挂载／存储三条规则没有执行记录：原件 JS 无法解析阻断 AST 规则；修复后 KPI 方法已接受 kpi、只有绑定未传参，watcher 为零参数加内部 settings 读取，初始化为 initDataImpl 调用且存在提前 localStorage 读取，均不匹配上一离线产物的窄形状。既不能说规则执行后失效，也不能说全部情况已闭环。generation Prompt 本轮未改，旧规则不保证新生成代码总是合法。

分析在 data/experiments/analysis/formal_30_v4_semantic_closure_v3_20261103_175401/REPORT.md、summary.json；可复核脚本 analyze_formal_30_v4_semantic_closure_v3.py，包含真实范围正文摘要及四项隔离检测证据。最终／拒收文件、签名、评估、summary、history 及 token 核对一致。

下一步建议（未实施）：暂停直接付费重跑，先在本冻结样本补覆盖状态感知的候选门正反例，区分旧缺陷首次可见与真实新增缺陷，同时保留真实回归保护；再审查非法标识符结构恢复、绑定缺参及有状态图表 helper 的显式依赖转换。不能全局豁免隐藏依赖，也不能靠放宽检测刷通过。新规则先离线验证，再建新版本。

本次分析没有修改运行期规则／Prompt／验证器、测试页面或原产物，没有再次调用模型或运行人工交互。TaskManager 人工通过候选保持；MonitoringDashboard 未进入人工验收；完整 30 项回归仍未完成，未 commit／push。

分析工具新增 3 项只读正反例通过（0.108 秒）：真实四项重现，主体变化与调用变化／额外诊断不能误认成相同缺陷。分析后快照 315 文件、协议 304 项 live 校验再次通过；没有把本次分析测试冒充新一轮完整运行期测试。

## 覆盖状态与图表 helper 通用闭环（2026-11-03，离线）

以冻结 v3 失败快照 315 文件为依据，本轮修改 repair 规则、候选门和底层分析模块，没有修改 generation／服务器 Prompt 或 repair Prompt 文本。新增源注册的非法标识符恢复：token 定位后只改声明与对应注册，严格重解析及 binding 验证；不改字符串／模板文字／注释，额外用途或来源不明不猜改。

候选门通过相同注册 tag、相同方法 AST 及渲染表达式／位置证明，区分既有缺陷首次可见与真正新增缺陷。四项旧图表诊断可证明为既有，新识别的零参数 watcher 缺失存储也在父 inited 不变和源证据下单独重检，共五项在当前检查器可解释。它们仍是 validation_errors，不自动判通过；旧 history 的四项原判据及分数不改。只把计算常量 2 改成 3，保护门仍拒收；未知来源、调用变化、其它新诊断也不豁免。

私有子组件 helper 将常量路径的只读 data.get 依赖改成参数，同时更新所有模板和实例调用，要求唯一父 components 注册、无其它公开引用、精确参数数目及自包含计算证明，再复用 computed helper 提取。拒绝未知 this／外部自由变量／写入／参数 mutation／裸方法引用等。源规则还补齐已等价方法的缺参绑定、零参数 watcher 的明确等价读取前缀，以及与已转换 self guard 并存的唯一非 self boolean 事件。模板编辑事务化，模板解析失败不再阻断独立脚本诊断。

最终完整 319/319 测试通过，142.262 秒，本轮新增 19 项。包括真实失败、真假回归、调用覆盖与拒修边界、真实 Stage 后处理、mock 调度；实际 San computed 在 0／1／7／30／90 点和 600／1200 宽度下比较折线与圆点坐标。没有浏览器交互测试或额外模型请求。一次开发中途修改私有注册边界的验证因哈希变化未封存，之后以最终代码完整重跑才封存。

离线证据包 data/experiments/analysis/formal_30_v4_coverage_helper_closure_v1_20261103，22 文件校验通过。v3 保留原件经 1 步后仍有 19 条／9 类其它诊断，初始 DOM 0.9781，仍失败；v3 拒收候选经 1 步后静态错误清零、DOM 0.9825、候选门通过，仍只是离线修复通过，不是新模型或人工验收通过。随机输入未固定，不挑选相似度。原件未完整闭合的事实保留。

详见原 v3 分析目录 COVERAGE_HELPER_FOLLOWUP_REPORT.md、新包 replay.json／unit_tests.txt。原始 v3 快照 315 文件和 TaskManager 候选 13 文件复核通过。原 v3 仍 0/1 通过、240,133 tokens 不变。未改测试页面、未 commit／push；完整 30 项回归仍未完成。

下一步需另建冻结协议／checkpoint，仅对 MonitoringDashboard 作新生成验证；不把离线文件当初始输出，不直接 --force 重抽旧批。旧 live 基线与新规则不同，不修改旧 SHA 清单来绕过。模型调用仍由操作者启动，新结果先冻结，再检查和人工验证。

## semantic closure v4 单样本协议冻结（2026-11-03，尚未调用模型）

在上一节离线闭环之后，建立新批次 formal_30_v4_semantic_closure_v4，仅 MonitoringDashboard；仍是重复开发回归，不是独立未见测试，也不表示完整 30 项回归已完成。TaskManager 排除，保留其已验收候选和独立验收文件；本次未切换两个测试页面。

协议路径 data/experiments/protocols/formal_30_v4_semantic_closure_v4_frozen_20261103，冻结时间 19:23:13（UTC+8）。包含 297 个冻结输入／代码／测试文件及证据、日志等，329 项 frozen/live SHA-256 校验通过。v3 原始快照 315 文件、离线包 22 文件、TaskManager 候选包 13 文件及其验收 SHA 全部复核；旧快照／协议不修改。

预运行完整 325/325 测试通过，168.182 秒；比上一节新增 6 项启动边界测试，包括仅一个样本、固定环境、独立 checkpoint、无 --force、先验 live 校验、校验或密钥失败不启动服务／批次。无模型只读重放再次完成：保留原件仍 19 条诊断、DOM 0.9781；捕获候选静态零错误、DOM 本次 0.9912。前次离线包的 0.9825 仍保留，不挑较高分替换；未固定随机输入。这不是本轮新生成或人工交互通过。

新配置和入口分别为 migration_pipeline/batch_configs/formal_30_v4_semantic_closure_v4.json、run_formal_30_v4_semantic_closure_v4.py；另有 prepare_formal_30_v4_semantic_closure_v4.py 与 freeze_formal_30_v4_semantic_closure_v4_results.py。相对 v3 冻结运行期，仍只有上一节披露的 4 个已有文件变化及新增辅助模块，没有在本步骤继续修改修复算法、generation 或服务端 Prompt。

输入策略 fresh_from_vue_not_prior_san：同一源 Vue 重新提取、生成，再沿本次候选链修复，不注入离线答案。最大 3 次 LLM repair，每次调用前最多 4 个确定性步骤；qwen/qwen-plus、temperature=0.2，服务端超时 600 秒、generation／repair 客户端各 660 秒，与 v3 相同。dry-run 确认默认输出／报告／summary 全新且冻结时不存在；不覆盖旧结果，不使用 --force。

操作：在项目根目录 cardmig 环境，旧服务 Ctrl+C 后，窗口 A 本地设置 LLM_API_KEY，再执行 python -m migration_pipeline.run_formal_30_v4_semantic_closure_v4 --server；窗口 B 执行同入口 --execute。两入口先核对 live 哈希。当前仅完成协议准备，model_request_performed=false；实际服务器环境和上游模型版本不由文件哈希独立证明。运行结束先冻结全部原始结果再诊断／人工验证，不直接重抽。

Notebook 作为协议冻结后的叙述追加，不纳入运行期 live 输入，避免后续记录结果必须改写旧协议。原 v3 仍 0/1、240133 tokens；未 commit／push。

## semantic closure v4 实际结果：自动通过但重复初始化待处理（2026-11-03）

用户完成本轮后先冻结 data/experiments/snapshots/formal_30_v4_semantic_closure_v4_results_20261103_204227，342 文件校验通过；冻结时协议 329 项 live 校验通过。运行 19:40:56–19:55:09（UTC+8），853.422 秒。仅 MonitoringDashboard：1 完成、1 自动通过、0 失败、0 执行异常。最终静态 0 错误／1 条 migration_hints_not_landed 警告，初始 DOM 相似度 0.9912、标签／文本均 1.0。interaction_eval_passed=null，尚未人工验收。

生成 106577、修复 166226（135591 + 30635）、合计 272803 tokens；2 个确定性步骤、2 次 LLM repair 分开计数，均接受，无回退。初始静态 28 错误／DOM 0；第一规范化 step 补 prop schema、解包 directive／methods、重命名业务 initData 后 24 错误；第二 step 显式 helper 依赖后 22 错误，4 条隐藏依赖减少为 2 条，不是规则独自全部修好。第一次模型修复剩生命周期调用 1 错误，后处理记录 watcher 存储及布尔事件等；第二次模型修复后基础错误清零。最终文件、各候选签名、summary／history 与用量交叉核对一致。

本轮候选门全部 semantic_coverage_assessment=not_needed，无非法标识符恢复记录，不能把本次成功当作可见性转换分支又被触发验证；该分支证据仍是上一离线包。最终文件 MonitoringDashboard_normalized_1_normalized_2_repaired_repaired.san，精确 SHA-256 e013c2925374ff3fb43e7236467ea7d094c0bc818fb36fd30d605c1caac211e3。

只读 AST 复核发现基础评估漏检：源父 mounted 第 614 行仅一次 this.initData()；第一修复仅父 inited 调 initDataImpl、attached 缺失；第二修复在 attached 第 444 行补调用，却保留 inited 第 441 行同一调用，最终重复且提前初始化。初始化会重建图表／统计／日志等并读取设置，未做浏览器观测，不虚构重复定时器等具体症状。现有挂载存在性检查、严格展开序列规则和候选门没有覆盖此包装调用的跨阶段次数契约，不能简单归因随机。

另记录第二轮删除源和第一轮已有的 requestNotificationPermission 方法；未证明可达性／具体运行影响，不据此断言某个当前操作必报错。自动原分数继续保留 1/1，不回写为失败或人工通过；人工状态为 not_started_lifecycle_review_required。本次通过不代表完整 30 项通过。

分析工具 migration_pipeline/analyze_formal_30_v4_semantic_closure_v4.py；记录 data/experiments/analysis/formal_30_v4_semantic_closure_v4_20261103_204227/REPORT.md、summary.json。新增 3 项只读审计测试通过，0.531 秒，覆盖真实重复、首轮提前调用与子组件／注释不误计数；不是重新完整跑运行期测试。原始快照和当前 live 基线保持不变。

下一步先以冻结产物补父初始化阶段／次数的源驱动检测、候选门正反例及有边界离线修复，不直接付费重跑或人工勾通过。不得全局删除 inited 调用，须证明源仅挂载一次且保留 watcher／其它效果。方法丢失另行审查。当前没有修改生产规则／Prompt、生成产物或测试页面；没有再次模型调用、浏览器验证、commit／push。TaskManager 已验收候选保持；完整 30 项开发回归和探索性评估继续等待。

## 源初始化阶段／次数通用闭环（2026-11-03，离线）

承接 v4 自动通过但重复初始化漏检，新增 utils/startup_phase_semantics.cjs，并修改 source_phase_semantics.cjs、bounded_semantics.cjs、stages/repair.py、utils/repair_scheduler.py、utils/repair_prompt.py。通过现有 ValidateStage 桥接新增 source_startup_phase_not_landed；确定性预处理和模型返回后处理共用规则，候选门纳入阶段／次数回归，聚焦 repair Prompt 明确后移而非仅追加第二次调用。未改 generation／服务器 Prompt；不是只改 Prompt，也不按组件名硬编码。

契约来自源导出父 mounted 首个无参包装方法及目标唯一同体别名，包装体仅已有无参方法调用。源其它早期 hook、其它引用、模板事件、动态访问、spread／mixins／重复 option、歧义／async 等拒绝猜修。只处理 inited／attached 直接无参调用，不承诺任意间接调用图等价。自动变更仅允许一次 inited 末尾调用，其前只保留唯一源命名 watcher 注册；附加效果、条件调用、多次调用、未知用途、attached 顺序不符均不删除。

冻结 v4 首轮修复仅提前调用，本次以 relocated_source_proven_startup_wrapper 将其后移；冻结最终件提前且重复调用，以 removed_source_proven_duplicate_startup 仅删除提前语句。两份各 1 步，验证均静态 0 错误、DOM 0.9912；修复后 inited 0／attached 1，固定点及反向回归拒收验证通过。既有迁移提示警告保留，初始 DOM 不是全部交互通过。

新增 15 项正反例，涵盖通用名称／别名、真实两份产物、拒修边界、验证器到 Stage、模型返回后处理、候选门、真实调度器零模型闭合。安装的 San 构造／直接 attached hook 测试确认构造后初始化 0、attached 后 1、watcher 仍能响应；不是浏览器人工验收。最终完整 343/343 测试通过，207.094 秒，比 v4 预运行 325 项多此前 3 项审计和本轮 15 项。测试／重放前后代码哈希一致。

证据 data/experiments/analysis/formal_30_v4_startup_phase_closure_v1_20261103，24 文件校验通过，含 18 份代码／测试、两份独立派生 San、完整测试日志、replay.json、302 项 live inventory 和验证脚本。最终件派生候选 SHA 6e585ea512971c5f9840e440ffb16c51acac17e813547bac86856b0aebc9aef5；首轮派生候选 SHA 2c7697c35bbf51069403473e13847fbe9f3aad7784888d02dd27aa3a7492c44b。没有覆盖 canonical 产物。

原 v4 快照 342 文件、旧协议 frozen 329 项复核通过；当前代码已不同，旧 live 不应再被当作通过基线，更不能改旧 SHA 清单绕过。原 v4 仍 1/1 自动通过、272803 tokens，不回填本次新检测结果或离线通过。被删除的通知权限方法仍需单独审查，不能宣称所有语义缺陷已清零。

详细说明在原 v4 分析目录 STARTUP_PHASE_FOLLOWUP_REPORT.md，可只读重放 python -m migration_pipeline.replay_formal_30_v4_startup_phase_closure。没有新模型请求或浏览器交互，没有修改测试页面、commit／push。后续模型调用须另建冻结协议，不能直接用旧 v4 命令复跑；TaskManager 已验收候选保持，完整 30 项开发回归／探索性评估仍未完成。

## 通知权限方法丢失审查：未发现内部调用，不等于接口等价（2026-11-03）

继续审查 requestNotificationPermission：源 methods 和 SSM body_full 均在；初始生成文本也有定义，但位于未展开的 Vue methods 包装内，不能视为当时已正确成为 San 实例方法。第一次模型修复后顶层方法保留，第二次模型修复删除；第二次 Prompt 中当前 San 仍有该方法且要求保留无关方法，所以不是提取或候选上下文缺失。启动阶段离线修复仅处理重复初始化，没有补回此接口。

只读 AST 审查精确选择导出父组件：源父脚本引用 0、模板指令／事件／插值引用 0，本次源扫描没有动态 this 访问或 this 别名等未决站点。mounted、settings watcher、saveSettings 均不调用权限申请方法。源 addAlert 在桌面通知开启且已 granted 时构造通知，最终 San 仍保留这个构造路径。这些是静态事实，未请求浏览器权限或实际验证通知；不能证明外部消费者从不调用该实例方法。

结论：不能把删除当作没有问题，也不能把现有设置界面必然报错当作已证实。源方法接口确实丢失，外部接口等价未证明；源本身没有的保存／挂载权限请求触发点不可擅自添加。当前验证主要约束事件处理器，候选门缺少全面方法保留契约。SSM inferred side effects 为空也不证明无副作用，因为 body_full 中有 Notification.requestPermission()。

审计工具 migration_pipeline/audit_formal_30_v4_notification_method.py；记录 data/experiments/analysis/formal_30_v4_notification_method_audit_v1_20261103/REPORT.md、summary.json。新增 5 项只读审计测试通过，5.698 秒，覆盖脚本／模板调用、动态别名不确定性、子组件隔离及真实冻结链；不是重跑完整 348 项。旧完整 343 项结果不改写。

原始模型快照 342 文件、启动阶段离线包 24 文件复核通过。未修改运行期规则／Prompt、canonical 产物、summary 或测试页面，未调用模型／浏览器权限，未 commit／push。建议下一步按保留源方法接口，补通用方法保留诊断／候选门和有证据的恢复；允许合法别名，拒绝猜测闭包／外部依赖，不新增 UI／生命周期调用。此实现边界待用户确认，本次只审查，尚未冻结下一轮模型协议。

## 源方法接口保留通用闭环（2026-11-03，用户确认后实施）

新增 utils/method_interface_semantics.cjs，检查导出父组件可调用面，不以整段代码出现名字代替。source_phase／bounded 桥接新增 method_contracts；ValidateStage 对普通无 mounted 源也检查；repair 预处理与模型返回后处理共用恢复；候选门纳入 source_method_interface_not_landed 与 source_method_trigger_added。repair Prompt 全路径保留源方法清单，聚焦路径携带报告方法的原 body／params，并禁止新增 UI／生命周期调用。未改 generation／服务端 Prompt 或预算。

支持直接方法、函数值、无重赋值且可静态解析的本地函数绑定，以及同名公开包装接口。initData 保留名映射仍交既有专用规则，不覆盖 San 状态工厂。自动恢复限于源／目标内部均无识别引用、覆盖无不确定性、目标同名 option 缺失的同步简单方法；保留源条件和参数，转换已确认状态读取。未知闭包／外部依赖、全局遮蔽、动态访问、写操作、复杂参数／控制流等不猜补；不覆盖已有不透明 option。证明范围为可调用面及受限恢复，不是任意已有方法体或所有子组件 API 等价。

通知方法按源 body 补回，原 capability／desktopNotifications 条件保留，未向保存、挂载、watcher 或按钮增加调用。16 项新增测试覆盖实际候选、通用名称、拒修、绑定／包装、权限桩、Stage 两入口及真实调度器零模型闭合。受控 Node 桩在未调用／条件关闭／能力缺失时均为 0，仅显式调用且原条件满足后为 1；没有申请真实浏览器权限。

解析恢复后的旧接口诊断通过源注册恢复虚拟旧件精确重检解释，但仍保留为验证错误，不能免检或自动判成功；新增接口丢失／显式触发仍拒绝。旧 Vue methods 包装保持诊断稳定并先解包，不在包装内外重复补方法。初始化规则与方法保留不再互相掩盖。

首次完整 364 项有 1 项失败，来自旧 TaskManager 事件探针测试的预期集合尚未包含新接口诊断。复核后只更新当前预期及 isOverdue／openTaskModal 位置断言，仍要求历史探针停止，没有放宽规则或改旧证据。相关 32 项通过后重新完整 364/364 测试通过，219.653 秒；计数为先前 343 加 5 项审计和本轮 16 项。代码在测试／重放前后哈希一致。

三份冻结输入各一步离线处理，静态均 0 错误、初始 DOM 均通过：首轮修复后移初始化，0.9912；最终原件去重并补方法，0.9825；启动已修候选只补方法，0.9912。固定点和反向保护通过，迁移提示警告保留。后两份代码 SHA 相同为 119360600912a8851ba01035ff639eb477aa7c6b63edaac20235b0fa05a6283e，但评估输入／环境未固定导致实测不同；不挑较高分覆盖，不对具体差异来源作单因素归因。

证据 data/experiments/analysis/formal_30_v4_method_interface_closure_v1_20261103，31 文件校验，含 24 份代码／测试、3 份派生 San、测试日志、replay.json、307 项 live inventory 和验证脚本。详见通知接口审查目录 METHOD_INTERFACE_FOLLOWUP_REPORT.md。旧原始快照 342、启动包 24、旧协议 frozen 329 项复核通过。原 v4 仍 1/1、272803 tokens；未覆盖 canonical 产物、summary 或测试页，未新增模型／浏览器交互／权限请求，未 commit／push。

本节闭合已暴露的问题，不是全部业务语义、独立测试或新生成成功。旧 TaskManager 人工确认不改写，也不冒充新接口契约下的全面复核。下一步应冻结新协议再安排新生成验证，不能沿用旧 v4 launcher 或改旧 SHA 绕过；完整 30 项回归仍待完成。

## 启动阶段与方法接口闭环后：新生成回归协议冻结（2026-11-04）

承接上一节用户确认，准备 formal_30_v4_semantic_closure_v5，仅 MonitoringDashboard。名称 v5 是 formal v4 内部语义闭环第五轮，不是独立正式测试版本。本次只新增配置、launcher、preparer、原始结果封存入口和 10 项协议边界测试，没有再次修改生产规则或生成／服务器 Prompt。

协议 data/experiments/protocols/formal_30_v4_semantic_closure_v5_frozen_20261104，2026-11-04T14:27:03+08:00 冻结；312 项 frozen payload，352 项 SHA-256 frozen/live 校验通过。核对旧模型原始快照 342、方法接口离线包 31、启动离线包 24、TaskManager 候选包 13 文件，307 项旧离线 live inventory 全部与当前代码一致；相对旧模型运行协议的 8 项已有文件变更逐项列入 manifest，另冻结本轮 5 个新增文件。

完整本地测试 374/374 通过，236.060 秒（先前 364 项加本轮 10 项）。新增边界测试覆盖单样本、3 次修复预算、独立 checkpoint、禁止 force／旧路径、调用前 live 校验、子进程环境、无密钥停止，以及离线证据不得新增权限触发、缺失接口、重复初始化或伪称模型／浏览器验收。测试中的服务／批处理分派均为 mock，不是启动真实模型任务。

冻结前重新无模型回放 first_repair、final_raw、startup_fixed_final 三份原输入；静态均零错误、DOM 均 0.9912，初始化规则及方法恢复规则按原预期触发，固定点和逆向拒绝仍通过。原离线包曾记录 final_raw 为 0.9825，本次观测单独保存 offline_replay_log.txt，不覆盖旧报告、不选择较高分改写原结果，也不据此归因具体运行差异。

模型运行尚未开始，人工交互也未开始。下一次从相同 Vue 重新提取／生成，不继承旧 San 或注入派生已知答案；最多 3 次 LLM repair，每次调用前最多 4 步确定性处理、总最多 12 步。固定 qwen-plus、temperature=0.2、服务 600 秒和客户端 660 秒；未固定随机种子，服务实际模型版本及旧进程是否停止不能仅靠文件哈希独立证明。TaskManager 排除并保留原用户验收，两个测试页未改。

操作者先停止旧服务器，在项目根目录 cardmig 环境的窗口 A 设置本地 LLM_API_KEY，再执行 python -m migration_pipeline.run_formal_30_v4_semantic_closure_v5 --server；窗口 B 执行同模块 --execute。不要附加 --force 或旧输出路径。两入口先做新协议 live 校验；失败须停止，不能改旧 SHA 清单绕过。模型完成先封存全部原始结果再诊断，封存入口 migration_pipeline.freeze_formal_30_v4_semantic_closure_v5_results。

没有新模型请求、真实浏览器权限申请、commit 或 push。旧 v4 自动 1/1、272803 tokens 不改写；本节是运行前准备证据，不是新生成通过、完整 30 项回归通过或独立测试通过。完整 30 样本最终规则回归与披露接触史的候选集探索性评估仍待后续安排。

## semantic closure v5：原始自动通过，另发现 watcher 持久化遗漏（2026-11-04）

用户完成运行后先封存：data/experiments/snapshots/formal_30_v4_semantic_closure_v5_results_20261104_150846，364 文件校验通过；协议 frozen/live 352 项仍通过，没有修改运行期规则、Prompt、canonical 产物或原 summary。运行 14:58:19—15:08:24，605.000 秒，仅 MonitoringDashboard；1 完成、1 自动通过，无执行异常。生成 107336、修复 134895、共 242231 tokens。

初始 16 条静态错误、San 渲染失败，DOM 0。确定性候选 1 尝试注册标识符／prop／指令／methods 修复，被候选门以 4 条隐藏响应式依赖拒绝；保留该候选及原拒绝理由，不擅自改为误拒。实际 LLM 修复 1 次，门接受但仍有 missing_self_click_guard，DOM 0.9912。随后确定性候选 2 恢复 self-click/native-drag 语义，被接受后静态零错误、DOM 0.9912、tag/text 均 1.0；保留 migration_hints_not_landed 警告。before_llm_call=2 只是处理位置，实际没有第二次 LLM 调用。

最终 MonitoringDashboard_repaired_normalized_2.san，精确 SHA-256 c449590f735d97cab2dc91ea24f8353fd1bba1be3464b1b814af24eefd3be168。summary、history、全部候选进度哈希、最终文件和 token 交叉一致。没有取较高分替换最后候选，也不把确定性拒绝计作修复成功。

前轮两处复核：源 initData 的 7 条无参调用在 attached 首部按原顺序各出现一次，inited 只注册 watcher，未复现提前／重复初始化。本轮为直接展开，不是 initDataImpl 包装器，startup_contracts 为空，不宣称包装器规则本轮触发。requestNotificationPermission 已是父组件方法，源条件保留，源／目标内部识别引用均零且无未决站点；没有新增 UI／挂载权限触发，也没有记录补方法规则触发。未申请真实权限。

继续只读审查发现遗漏：Vue settings watcher 第 608 行 localStorage.setItem('dashboard_settings', JSON.stringify(newVal)) 在 San 第 638 行注册的 (newValue, oldValue) 回调中丢失，导出父组件亦没有其他直接 setItem 写入补偿，loadSettings 仍读取。现有 repairWatchPersistence 要求目标单参数与源同名，或特定无参局部取值形式；改名并带第二参数使其 continue，检测与修复一同跳过。因此当前 findings=[]、旧自动零错误均保留，但新增独立缺陷，不能称业务等价或所有闭环完成。浏览器刷新后的具体症状尚未测，不虚构已观察结果。

审计工具 migration_pipeline/analyze_formal_30_v4_semantic_closure_v5.py；分析目录 data/experiments/analysis/formal_30_v4_semantic_closure_v5_20261104_150846，含 REPORT.md、summary.json。6 项只读审计测试最终通过，2.502 秒，覆盖真实产物、参数差异、存储写入保留、子组件隔离、直接初始化次数和父导出歧义。初次歧义夹具误拼两个 SFC，修为同一 script 两个导出后重跑；未为通过测试放宽审计或生产规则。不是重新完整跑 380 项，旧完整 374 项记录不改写。

人工状态 not_started_watch_persistence_review_required，自动交互评估 null。未更换测试页、未再次调用模型、未 commit/push。建议下一步先基于冻结产物补 watcher 参数作用域 alpha-mapping／无用第二参数的有界检测、恢复、候选门和拒修反例；不能安全修也应明确诊断，不能继续静默视为通过。尚未实施生产修复，不直接付费重跑或进入独立测试；完整 30 项最终规则回归仍未完成。

## watcher 参数变化与持久化：通用有界闭环（2026-11-04，用户确认后实施）

新增 utils/watch_persistence_semantics.cjs，由 source_phase_semantics.cjs 调用，bounded_semantics.cjs 输出 watcher_contracts。复用已有 source_watch_persistence_not_landed 诊断、restored_source_watch_persistence 恢复及候选门登记；ValidateStage、确定性预处理与模型返回后处理共用规则，没有按组件／字段／存储键特判，也没有放宽门禁。

支持可证明源单参数 watcher 尾部 setItem(字面量键, JSON.stringify(当前值)) 与目标唯一直接父 inited 回调。检测比修复更宽：复杂参数、旧值实际使用、主体改动等保留诊断而不猜修。自动修复用词法绑定 alpha-mapping，支持改名当前值、未使用旧值参数或特定无参 const 取值；验证全局未遮蔽、绑定未重赋值、转换后的原主体逐项同序一致。额外效果、不同条件、复杂控制流、闭包／eval／arguments 等拒修。不是继承、多个注册、存储别名或任意 watcher 等价证明。

仅追加源尾部存储写入，并使用目标当前值绑定，保留现有参数、条件、刷新逻辑、模板／CSS／子组件。不会新增保存或挂载触发，也不会重复追加已有受支持写入。repair Prompt 增加上述参数／顺序约束；同时发现本例 SSM watch.handler_body 是截断预览，故 repair.py 传入 Vue 原文，由 AST 提取完整 handler 补充源证据，不依赖截断预览猜测持久化。未改 generation 或服务器 Prompt。

新增 17 项覆盖通用名称、参数／作用域、拒修、固定点、安装的 San 存储桩、Stage 两入口、候选门和实际调度器。受控 Node 桩构造后写入 0，设置变化两次后写入 2，不是真实浏览器存储。真实调度器 0 模型、1 确定性步骤闭合。首次完整 397 项有 2 个旧测试集合预期失败，246.666 秒；源码确认两份历史产物也缺少写入，仅增加当前诊断及 script.watch.settings 位置断言。两项单独通过后重新完整 397/397 通过，239.652 秒；旧快照和计分不变。v5 审计也区分冻结零错误与实时新诊断，不再强求旧漏检复现。

基于冻结 v5 最终件，修复前静态 1 条持久化错误、DOM 0.9912；一步仅追加 localStorage.setItem("dashboard_settings", JSON.stringify(newValue)) 后静态零错误、DOM 0.9825 仍通过。完整文本扣除这条插入与输入一致，固定点和正向门通过，逆向删除写入拒绝；初始化展开序列及通知权限接口／零新增引用继续成立。保留两次 DOM 实测，不选择高分或虚构视觉改善，不单凭分差归因具体运行差异。

离线证据 data/experiments/analysis/formal_30_v4_watch_persistence_closure_v1_20261104，于 2026-11-04T15:48:36+08:00 冻结，16 文件校验通过，含 11 份代码／测试、派生 San、完整测试日志、replay.json、317 项 live inventory 和校验脚本。派生候选 MonitoringDashboard_watch_persistence_offline.san，SHA-256 3cbe7d15e64ce8df02cd79f2dbd510af998a3f8bb05187df60d2c221926aa5b4。详见原 v5 分析目录 WATCH_PERSISTENCE_FOLLOWUP_REPORT.md。

原始快照 364、旧协议 frozen 352 项保持完整；当前生产代码已变，不能用旧 live 协议发起新模型调用，更不能改旧 SHA 绕过。原 v5 自动 1/1、242231 tokens 不回填离线结果。未改 canonical San 或人工页，没有新模型请求、真实浏览器交互／权限申请、commit 或 push。尚无新生成／人工通过；完整 30 项最终规则回归仍未完成。后续可人工确认独立离线候选，再另建当前规则的整批开发回归协议。

## MonitoringDashboard 离线持久化修复候选：用户人工确认（2026-11-04）

两个测试页切换为 monitoring-dashboard、props={}：Vue 为 v5 原始快照中的冻结源，San 为独立离线候选 MonitoringDashboard_watch_persistence_offline.san。只改组件配置，加载器、周围页面代码与冻结组件内容未改。San 候选 SHA-256 3cbe7d15e64ce8df02cd79f2dbd510af998a3f8bb05187df60d2c221926aa5b4；Vue SHA-256 f669fc59619a30ce94cf8188b57124ae7e7084aa65d84604751391e73ab0ae4b。

在要求重点检查 San 设置保存后刷新是否保留、期间避免 Vue 页共享存储干扰后，用户回复“通过”。据此仅将该精确派生候选记为 passed_by_user；用户未提供逐项操作或存储测量日志，不将测试要求反推为独立观察，也不虚构浏览器通知／权限结果。助手没有执行浏览器测试。

独立验收记录 data/experiments/analysis/formal_30_v4_semantic_closure_v5_20261104_150846/MonitoringDashboard_manual_acceptance_20261104.json，记录时间 2026-11-04T16:17:01+08:00，同时记录两个 runner 的哈希。离线包 16、原始快照 364 文件再次校验通过，旧冻结证据没有回写人工状态。

原 v5 自动 1/1、242231 tokens 与原始未补写入产物不改写；这不是该原件人工通过、当前规则新生成通过、独立测试通过或完整 30 项通过。此次仅记录用户验收、追加本节，未改生产规则、测试页面或启动模型，未 commit/push。下一步应为当前最终规则准备全新 30 样本开发回归协议及输出目录，先冻结并校验，再由用户运行；不能使用旧 v5 live 协议或旧 summary checkpoint 代替整批回归。

## 转入最终规则完整 30 样本回归：协议冻结（2026-11-04）

用户确认后准备 formal_30_v4_full_regression_v2，沿用原顺序／Vue 字节及 10/10/10 分层，TaskManager 和 MonitoringDashboard 都从 Vue 重新生成，不继承验收 San 或旧 checkpoint。仍为已接触样本的开发回归，非独立测试。每样本最多 3 次 LLM repair、每次前最多 4 步确定性处理、每样本总最多 12 步，residual_v2；环境沿用 600／660 秒，不与旧混合尝试口径混为一谈。

协议 data/experiments/protocols/formal_30_v4_full_regression_v2_frozen_20261104，2026-11-04T16:35:26+08:00 冻结，322 项 payload、351 项 frozen/live 哈希通过。完整 407/407 测试通过，245.997 秒；无模型闭环回放静态／DOM 通过，本次 0.9912 另存，旧 0.9825 不改写。30 项 dry-run 和新输出路径匹配；模型尚未运行，输出／checkpoint 冻结时不存在。

详细运行前记录和后续整批结果转入 [2026-11-04_formal_30_v4_full_regression_v2.ipynb](2026-11-04_formal_30_v4_full_regression_v2.ipynb)。保留旧 v3 18/30、首次 v4 25/30，以及两份独立人工验收，不拼接成新批结论。本步骤未改生产规则、人工页、旧产物或计分，未启动服务／调用模型，未 commit/push；等待用户按新 launcher 启动整批，完成后先封存再分析。